In [ ]:
import gmsh
import numpy as np
import pyvista
import ufl

from mpi4py import MPI
from petsc4py import PETSc

from dolfinx import fem, io, plot
from dolfinx.io import gmshio
from dolfinx.plot import vtk_mesh
from dolfinx.mesh import locate_entities_boundary
from dolfinx.fem import (
    Constant,
    Function,
    Expression,
    functionspace,
    dirichletbc,
    locate_dofs_topological,
    form
)
from dolfinx.fem.petsc import (
    LinearProblem,
    assemble_vector,
    assemble_matrix,
    create_vector,
    apply_lifting,
    set_bc
)

from ufl import (
    TrialFunction,
    TestFunction,
    SpatialCoordinate,
    grad,
    dot,
    dx,
    ds,
    as_vector
)

In [ ]:

# Parâmetros gerais do problema

gdim = 2
model_rank = 0
mesh_comm = MPI.COMM_WORLD

# Geometria do fusível - unidades em metros

# Dimensões aproximadas usadas nos exemplos antigos
x_rectangle = 0.0052      # comprimento de cada placa metálica [m]
y_rectangle = 0.0100      # altura de cada placa metálica [m]
x_gap = 0.0041            # distância entre as placas [m]

# Filamento central
fuse_thickness = 0.0005   # espessura/altura do filamento no plano 2D [m]

# Posição vertical do filamento
y_half = y_rectangle / 2
y_fuse = y_half - fuse_thickness / 2

# Comprimento total do domínio metálico
x_total = 2*x_rectangle + x_gap

# Espessura fora do plano
# Como o problema é 2D planar, assumimos que não varia na profundidade.
# A espessura real entra depois para converter corrente/área/volume.
thickness_z = 0.0005  # [m]


# Propriedades do estanho
# Valores usados como base nos notebooks antigos.
# Vamos conferir/ajustar depois conforme a fonte que você usar no relatório.

rho_e0 = 11e-8              # resistividade elétrica de referência [ohm.m]
sigma0 = 1 / rho_e0         # condutividade elétrica [S/m]

rho = 7310                  # densidade [kg/m^3]
cp = 227                    # calor específico [J/(kg.K)]
k_thermal = 66.6            # condutividade térmica [W/(m.K)]

T_amb = 25 + 273.15         # temperatura ambiente [K]
T0_ref = T_amb              # temperatura de referência [K]
T_fusao = 232 + 273.15      # temperatura de fusão do estanho [K]

# Coeficiente de temperatura da resistividade.
# Deixe como parâmetro para ajustarmos com a fonte do relatório.
alpha_rho = 4.5e-3          # [1/K] valor aproximado, confirmar no relatório


# Corrente nominal do fusível
# Vamos começar com 5 A, como nos exemplos antigos.
# Depois você pode trocar para a corrente nominal escolhida do fusível.
I_nominal = 5.0             # [A]

# Parâmetros temporais para o problema térmico

t = 0.0
T_final = 10.0              # tempo final [s]
num_steps = 100
dt = T_final / num_steps

In [ ]:
# ============================================================
# Criação da geometria com GMSH
# ============================================================

gmsh.initialize()
gmsh.model.add("fusivel_automotivo_2d")

# Placa esquerda
rec_left = gmsh.model.occ.addRectangle(
    0.0, 0.0, 0.0,
    x_rectangle, y_rectangle
)

# Placa direita
rec_right = gmsh.model.occ.addRectangle(
    x_rectangle + x_gap, 0.0, 0.0,
    x_rectangle, y_rectangle
)

# Filamento central
fuse_bridge = gmsh.model.occ.addRectangle(
    x_rectangle, y_fuse, 0.0,
    x_gap, fuse_thickness
)

gmsh.model.occ.synchronize()

# Faz a união das três regiões metálicas
fused, _ = gmsh.model.occ.fuse(
    [(2, rec_left)],
    [(2, rec_right), (2, fuse_bridge)],
    removeObject=True,
    removeTool=True
)

gmsh.model.occ.synchronize()

# Recupera as superfícies resultantes
surfaces = gmsh.model.getEntities(dim=2)
surface_tags = [tag for dim, tag in surfaces]

# Marca a região metálica do fusível
FUSE_DOMAIN = 1
gmsh.model.addPhysicalGroup(2, surface_tags, FUSE_DOMAIN)
gmsh.model.setPhysicalName(2, FUSE_DOMAIN, "FuseDomain")

In [ ]:
# ============================================================
# Marcação das fronteiras
# ============================================================

LEFT_CONTACT = 11
RIGHT_CONTACT = 12
OUTER_BOUNDARY = 13

# Pega todas as curvas de fronteira das superfícies
boundary_curves = gmsh.model.getBoundary(
    [(2, tag) for tag in surface_tags],
    oriented=False,
    recursive=False
)

left_curves = []
right_curves = []
outer_curves = []

tol = 1e-9

for dim, tag in boundary_curves:
    cx, cy, cz = gmsh.model.occ.getCenterOfMass(dim, tag)
    
    if np.isclose(cx, 0.0, atol=tol):
        left_curves.append(tag)
    elif np.isclose(cx, x_total, atol=tol):
        right_curves.append(tag)
    else:
        outer_curves.append(tag)

gmsh.model.addPhysicalGroup(1, left_curves, LEFT_CONTACT)
gmsh.model.setPhysicalName(1, LEFT_CONTACT, "LeftContact")

gmsh.model.addPhysicalGroup(1, right_curves, RIGHT_CONTACT)
gmsh.model.setPhysicalName(1, RIGHT_CONTACT, "RightContact")

gmsh.model.addPhysicalGroup(1, outer_curves, OUTER_BOUNDARY)
gmsh.model.setPhysicalName(1, OUTER_BOUNDARY, "OuterBoundary")

gmsh.model.occ.synchronize()

In [ ]:
# ============================================================
# Controle do tamanho da malha
# ============================================================

mesh_size = 2.0e-4  # tamanho máximo dos elementos [m]

gmsh.option.setNumber("Mesh.CharacteristicLengthMax", mesh_size)
gmsh.option.setNumber("Mesh.CharacteristicLengthMin", mesh_size / 5)

gmsh.model.mesh.generate(2)

In [ ]:
# ============================================================
# Conversão da malha do GMSH para Dolfinx
# ============================================================

domain, cell_tags, facet_tags = gmshio.model_to_mesh(
    gmsh.model,
    mesh_comm,
    model_rank,
    gdim=gdim
)

domain.name = "fusivel"
cell_tags.name = "cell_tags"
facet_tags.name = "facet_tags"

gmsh.finalize()

In [ ]:
# ============================================================
# Visualização rápida da malha
# ============================================================

pyvista.start_xvfb()

domain.topology.create_connectivity(domain.topology.dim, domain.topology.dim)

grid = pyvista.UnstructuredGrid(*vtk_mesh(domain, domain.topology.dim))

plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True)
plotter.view_xy()
plotter.camera.zoom(1.2)
plotter.show()

In [ ]:
# ============================================================
# Exportação da malha para XDMF
# ============================================================

with io.XDMFFile(domain.comm, "tc06_malha.xdmf", "w") as xdmf:
    xdmf.write_mesh(domain)